# 1. Frontend performance — Cloudflare Pages

The site is a static export (41 files, ~936 KB) served from Cloudflare's edge.
There is no server render, so the questions worth asking are narrow:

1. **Time to first byte**, cold and warm — does the edge cache actually serve us?
2. **Transfer size** — is compression on, and is the JS budget sane?
3. **Lighthouse** — the synthetic score, for comparability with other sites.

TTFB and sizes need nothing but network access. Lighthouse needs a
PageSpeed Insights key (`PSI_API_KEY`); without one the request shares a
global quota that is normally exhausted.

In [ ]:
import json, os, statistics, time, urllib.request, gzip, io
import matplotlib.pyplot as plt
import pandas as pd

SITE = os.environ.get("SITE_URL", "https://esco.lucasain.dev")
REPEATS = 12

print("target:", SITE)

## 1.1 Time to first byte

`REPEATS` sequential requests. The first is expected to be slower: a cold
edge cache, plus TLS and connection setup that later requests may reuse.
We record Cloudflare's own `cf-cache-status` header so a slow sample can be
attributed rather than guessed at.

In [ ]:
def timed_get(url, headers=None):
    req = urllib.request.Request(url, headers=headers or {})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=30) as r:
        first = time.perf_counter() - t0          # header arrival ~= TTFB
        body = r.read()
        total = time.perf_counter() - t0
        return {
            "ttfb_ms": first * 1000,
            "total_ms": total * 1000,
            "bytes": len(body),
            "cache": r.headers.get("cf-cache-status", "-"),
            "encoding": r.headers.get("content-encoding", "none"),
        }

samples = [timed_get(SITE) for _ in range(REPEATS)]
df = pd.DataFrame(samples)
df.index.name = "request"
display(df.round(1))

warm = df.iloc[1:]
print(f"cold  TTFB : {df.ttfb_ms.iloc[0]:.0f} ms")
print(f"warm  TTFB : median {warm.ttfb_ms.median():.0f} ms, "
      f"p95 {warm.ttfb_ms.quantile(0.95):.0f} ms")
print(f"cache statuses seen: {df.cache.value_counts().to_dict()}")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))

a.plot(df.index, df.ttfb_ms, marker="o", label="TTFB")
a.plot(df.index, df.total_ms, marker="s", alpha=.6, label="full response")
a.set_xlabel("request #"); a.set_ylabel("ms")
a.set_title("Latency across sequential requests")
a.legend(); a.grid(alpha=.3)

b.boxplot([df.ttfb_ms.iloc[1:]], tick_labels=["warm TTFB"])
b.scatter([1], [df.ttfb_ms.iloc[0]], color="crimson", zorder=3, label="cold (1st)")
b.set_ylabel("ms"); b.set_title("Cold vs warm"); b.legend(); b.grid(alpha=.3)

fig.tight_layout()

## 1.2 Transfer size and compression

A static export's cost is almost entirely its assets. Here we walk the HTML,
pull every `_next/static` reference, and record compressed versus
uncompressed size. Uncompressed JS is what the browser must *parse*, which is
the part that shows up as Total Blocking Time.

In [ ]:
import re

html = urllib.request.urlopen(SITE, timeout=30).read().decode()
assets = sorted(set(re.findall(r'/_next/static/[A-Za-z0-9_./-]+\.(?:js|css)', html)))
print(f"{len(assets)} static assets referenced")

rows = []
for path in assets:
    req = urllib.request.Request(SITE.rstrip("/") + path,
                                 headers={"Accept-Encoding": "gzip, br"})
    with urllib.request.urlopen(req, timeout=30) as r:
        raw = r.read()
        enc = r.headers.get("content-encoding", "none")
    # Decompress to get the parse-time size, not just the wire size.
    if enc == "gzip":
        plain = gzip.decompress(raw)
    else:
        plain = raw   # br/none: report wire size and flag it
    rows.append({"asset": path.split("/")[-1], "kind": path.rsplit(".", 1)[-1],
                 "wire_kb": len(raw) / 1024, "parsed_kb": len(plain) / 1024,
                 "encoding": enc})

sizes = pd.DataFrame(rows).sort_values("wire_kb", ascending=False)
display(sizes.round(1))
print(f"total on the wire : {sizes.wire_kb.sum():.0f} KB")
print(f"total to parse    : {sizes.parsed_kb.sum():.0f} KB")
print(f"page HTML         : {len(html)/1024:.1f} KB")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))

top = sizes.head(8).iloc[::-1]
a.barh(top.asset.str.slice(0, 26), top.wire_kb)
a.set_xlabel("KB on the wire"); a.set_title("Largest assets")
a.grid(alpha=.3, axis="x")

by_kind = sizes.groupby("kind")[["wire_kb", "parsed_kb"]].sum()
by_kind.plot(kind="bar", ax=b, rot=0)
b.set_ylabel("KB"); b.set_title("Wire vs parsed, by type")
b.grid(alpha=.3, axis="y")

fig.tight_layout()

## 1.3 Lighthouse

Google's hosted Lighthouse, so no local Chrome is needed. Set `PSI_API_KEY`
to avoid the shared keyless quota — a free key from the Google Cloud console
allows 25k requests/day.

Mobile is the stricter of the two strategies and the one worth reporting: it
models a slower CPU and network.

In [ ]:
PSI = "https://www.googleapis.com/pagespeedonline/v5/runPagespeed"
KEY = os.environ.get("PSI_API_KEY", "")

def lighthouse(url, strategy="mobile"):
    q = f"{PSI}?url={url}&strategy={strategy}&category=performance"
    if KEY:
        q += f"&key={KEY}"
    with urllib.request.urlopen(q, timeout=120) as r:
        d = json.load(r)
    if "error" in d:
        raise RuntimeError(d["error"].get("message", "unknown PSI error"))
    lr = d["lighthouseResult"]
    audits = lr["audits"]
    keep = ["first-contentful-paint", "largest-contentful-paint",
            "total-blocking-time", "cumulative-layout-shift", "speed-index"]
    return {
        "strategy": strategy,
        "score": round(lr["categories"]["performance"]["score"] * 100),
        **{k: audits[k]["numericValue"] for k in keep if k in audits},
    }

results = []
for strategy in ("mobile", "desktop"):
    try:
        results.append(lighthouse(SITE, strategy))
        print(f"{strategy}: score {results[-1]['score']}")
    except Exception as exc:
        print(f"{strategy}: SKIPPED - {exc}")
        print("  (set PSI_API_KEY, or run `npx lighthouse` locally with Chrome installed)")

lh = pd.DataFrame(results)
if not lh.empty:
    display(lh.round(0))

In [ ]:
if not lh.empty:
    metrics = [c for c in lh.columns if c not in ("strategy", "score")]
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))

    a.bar(lh.strategy, lh.score, color=["#888", "#444"])
    a.set_ylim(0, 100); a.set_ylabel("performance score")
    a.set_title("Lighthouse score"); a.grid(alpha=.3, axis="y")
    for i, v in enumerate(lh.score):
        a.text(i, v + 2, str(v), ha="center")

    x = range(len(metrics))
    w = 0.38
    for i, (_, row) in enumerate(lh.iterrows()):
        b.bar([v + i * w for v in x], [row[m] for m in metrics], w, label=row.strategy)
    b.set_xticks([v + w / 2 for v in x])
    b.set_xticklabels([m.replace("-", "\n") for m in metrics], fontsize=7)
    b.set_ylabel("ms (CLS is unitless)"); b.set_title("Core metrics")
    b.legend(); b.grid(alpha=.3, axis="y")
    fig.tight_layout()
else:
    print("no Lighthouse data - see the message above")

## What to take from this

Fill in after running. The things worth writing down:

- **Warm TTFB** is the number that represents the edge. If it is not well
  under 100 ms, the cache is not doing its job — check `cf-cache-status`.
- **Cold vs warm gap** is TLS plus cache miss. A large gap on a static site
  usually means low traffic rather than a problem.
- **Parsed KB** matters more than wire KB for interactivity. The graph view
  is hand-rolled SVG precisely to keep this small: no d3, no chart library.
- A near-100 Lighthouse score is expected for a static export and is not
  evidence of much. The interesting comparison is *before and after* adding
  a dependency.